# 04 - Strings and Operations

In this notebook:
- Learn Kamailio cfg string operations
- Understand automatic type conversion between integer and string
- Deep dive into `$var` vs `$avp` differences
- Practice string formatting and variable substitution

---

## Data Types in Kamailio

Kamailio cfg uses **dynamic typing**. The type is determined automatically when a value is assigned:
- `"hello"` → string
- `42` → integer
- The type is determined at assignment time and can change later

## 1. String Basics — Assignment and Output

In [ ]:
$var(greeting) = "Hello";
$var(target) = "Kamailio";
xlog("$var(greeting), $var(target)!");

## 2. String Concatenation (+)

The `+` operator concatenates strings.

In [ ]:
$var(user) = "1001";
$var(domain) = "example.com";
$var(uri) = "sip:" + $var(user) + "@" + $var(domain);
xlog("Constructed URI: $var(uri)");

## 3. Integer Arithmetic

The `+` operator also performs addition. If all operands are integers, it adds; if any is a string, it concatenates.

In [ ]:
$var(a) = 10;
$var(b) = 20;
$var(sum) = $var(a) + $var(b);
xlog("Sum: $var(a) + $var(b) = $var(sum)");

## 4. Variable Substitution with xlog

`xlog()` automatically substitutes `$variables` inside `"..."`. This is the foundation of Kamailio debugging.

In [ ]:
%%sip INVITE
From: <sip:alice@example.com>;tag=abc123
To: <sip:bob@example.com>

In [ ]:
$var(caller) = $(fu{uri.user});
$var(callee) = $(tu{uri.user});
xlog("Caller: $var(caller), Callee: $var(callee)");
xlog("Request URI: $ru");
xlog("Call-ID: $ci");

## 5. $var vs $avp — When to Use Which

| Aspect | `$var` | `$avp` |
|--------|--------|--------|
| Scope | Per-process (not shared) | Per-transaction (shared) |
| Storage | Single value | Stack (multiple values) |
| Performance | Fast | Relatively slower |
| Use case | Temporary calculations, flags | Session data, caller info |

Practical tip: Use `$var` for simple calculations or temporary values, `$avp` when data needs to be shared across transactions.

In [ ]:
# $var: simple counter
$var(retry_count) = 0;
$var(retry_count) = $var(retry_count) + 1;
xlog("Retry count: $var(retry_count)");

In [ ]:
# $avp: transaction data
$avp(caller_domain) = $(fd{uri.host});
xlog("Caller domain stored in AVP: $avp(caller_domain)");

## 6. String Manipulation with Transforms

Transform functions like `{s.upper}`, `{s.lower}`, `{s.len}` manipulate strings.

In [ ]:
$var(name) = "alice";
xlog("Original: $var(name)");
xlog("Upper: $(var(name){s.upper})");
xlog("Lower: $(var(name){s.lower})");
xlog("Length: $(var(name){s.len})");

## 7. Extracting Parts from URIs

Extracting user, host, and port from SIP URIs is very common in practice.

In [ ]:
$var(raw_uri) = "sip:1001@10.0.0.1:5060";
xlog("Full URI: $var(raw_uri)");
xlog("User: $(var(raw_uri){uri.user})");
xlog("Host: $(var(raw_uri){uri.host})");
xlog("Port: $(var(raw_uri){uri.port})");

## 8. Comparison Operations for Routing Decisions

String/integer comparisons determine routing branches.

In [ ]:
%%sip INVITE
From: <sip:1001@example.com>;tag=x1
To: <sip:2001@example.com>

In [ ]:
$var(dest) = $(ru{uri.user});

if ($var(dest) == "2001") {
    xlog("VIP customer — priority routing");
} else {
    xlog("Normal customer — standard routing");
}

---

### Summary

| Concept | Syntax |
|---------|--------|
| String assignment | `$var(x) = "hello";` |
| Integer assignment | `$var(n) = 42;` |
| String concatenation | `"a" + "b"` or `$var(x) + $var(y)` |
| Integer addition | `$var(a) + $var(b)` (both integers) |
| Variable substitution | `xlog("value: $var(x)");` |
| Case conversion | `$(var(x){s.upper})`, `$(var(x){s.lower})` |
| Length | `$(var(x){s.len})` |
| URI decomposition | `{uri.user}`, `{uri.host}`, `{uri.port}` |
| Equality check | `==`, `!=` |

Next notebook: **Intermediate/01-transformations.ipynb** →